# 9 · Semantic Ctrl-F: find by meaning, answer through chat

## The problem

Browser Find is a substring match. Search a help page for *"cancel"* and you miss *"You can **terminate your
subscription** at any time"*. Users don't know the page's vocabulary; they ask *"how do I stop getting charged every
month?"*, and Ctrl-F finds nothing.

Pasting the whole page into a chatbot fixes the vocabulary problem but creates a new one: when the page doesn't contain
the answer, the chatbot often answers anyway, from its general knowledge, in a confident voice.

## Why Jev

Two jobs, two models, each doing what it's best at:
- **Jev retrieves and gates.** A `Choice` picks the line that best answers the query, and a `Noul` decides whether the
  page answers it **at all**. Both are typed, calibrated judgments in one ~100 ms, sub-cent call.
- **OpenAI phrases.** Only if Jev says the answer is on the page does an LLM write a one-sentence reply, and it is
  handed **only the winning line**, so it can't wander off the page.

> **Jev decides where to look, OpenAI just says it nicely.**

## What we'll build

```
query + page lines ──► Jev, ONE call ──────────────► answerable < 0.5 ? ──yes──► "not on this page" (no LLM call)
                        • answerable (Noul)                   │ no
                        • best line  (Choice over L0..L7)     ▼
                                                     OpenAI: "using ONLY this line, answer in one sentence"
```

## What you'll learn
- How a `Choice` can range over **data** (the lines of a page), not just fixed categories.
- Why a `Choice` needs a separate `Noul` gate: a `Choice` **always** picks something.
- The *retrieve-then-generate* split: the cheap judge decides, the LLM only phrases what it's handed.
- This is the core loop of a "semantic Ctrl-F" Chrome extension.

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API keys from the `.env` file at the repo root (`TYPESAFE_API_KEY` for Jev, `OPENAI_API_KEY` for OpenAI),
- creates **two clients**: `jev` does the retrieval and gating, `oai` writes the final sentence,
- imports the two Jev question types we'll use, `Noul` and `Choice`.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from typesafe_sdk import TypeSafeClient, Noul, Choice

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: which line, and is the answer here at all?
oai = OpenAI()           # the writer: turns the winning line into a one-sentence reply

print("clients ready")

clients ready


## Step 1 · The page we search

A short help page for a made-up whiteboard app, one entry per line. In a browser extension, these would be the text
blocks scraped from the current tab.

Notice that the questions we'll ask share almost **no keywords** with the lines that answer them.

In [2]:
PAGE = [
    "Welcome to Nimbus, the collaborative whiteboard for remote teams.",
    "You can terminate your subscription at any time from Settings > Billing.",
    "Boards you create are private until you invite collaborators by email.",
    "Export any board to PDF or PNG from the share menu in the top right.",
    "We never sell your data; see our privacy policy for details.",
    "Nimbus works in any modern browser; there is no separate app to install.",
    "The Free plan includes three boards; paid plans remove the limit.",
    "Need a hand? Reach support in-app or at help@nimbus.example.",
]
for i, line in enumerate(PAGE):
    print(f"L{i}  {line}")

L0  Welcome to Nimbus, the collaborative whiteboard for remote teams.
L1  You can terminate your subscription at any time from Settings > Billing.
L2  Boards you create are private until you invite collaborators by email.
L3  Export any board to PDF or PNG from the share menu in the top right.
L4  We never sell your data; see our privacy policy for details.
L5  Nimbus works in any modern browser; there is no separate app to install.
L6  The Free plan includes three boards; paid plans remove the limit.
L7  Need a hand? Reach support in-app or at help@nimbus.example.


## Step 2 · Jev retrieves: is it here, and which line?

We ask **two questions in one call**:

| Key | Type | Returns | Role |
|---|---|---|---|
| `answerable` | `Noul` | `.noul`, P(yes), 0 to 1 | **the gate**: does any line answer the query? |
| `best` | `Choice` | `.choice`, `.confidence`, `.probabilities` | **the pointer**: which line answers it best? |

**`Noul`** is a yes/no question answered as a probability: `.noul` is P(yes).

**`Choice`** picks one label from `criteria`, a dict of `label → description`. It returns `.choice` (the winner),
`.confidence` (how sure Jev is), and `.probabilities` (a dict with a probability for **every** label). Here the labels
are `L0`..`L7` and each description **is the line itself**, so the options are built from the data at run time.

**Why we need both.** A `Choice` must pick one of its options, even when none of them fits. Ask *"is there a student
discount?"* and it will still point at *some* line. The `Noul` is what lets the system say *"not on this page"*.

The `state` is a dict, `{"query": ..., "page": [...]}`, and the backtick names in the instructions (`` `query` ``,
`` `page` ``) refer to its keys.

This cell only **defines** `retrieve()` and the gate threshold. Nothing is sent to Jev yet.

In [3]:
ANSWERABLE_AT = 0.5   # below this, the page doesn't answer the query: say so and skip the LLM


def retrieve(query, client):
    if "PAGE" not in globals():
        raise RuntimeError("PAGE is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(
        state={"query": query, "page": PAGE},
        questions={
            "answerable": Noul(instructions="Does any line in `page` contain an answer to `query`?"),
            "best": Choice(
                instructions="Which line of `page` best answers `query`?",
                criteria={f"L{i}": line for i, line in enumerate(PAGE)},   # one option per line
            ),
        },
    )
    best = r.choices["best"]
    return int(best.choice[1:]), r.nouls["answerable"].noul, best.confidence   # "L3" -> 3

Now the raw retrieval for two queries: one the page answers, and one it doesn't.

In [4]:
for q in ["how do I stop getting charged every month?", "is there a student discount?"]:
    idx, answerable, conf = retrieve(q, jev)
    print(f"{q}\n   answerable {answerable:.2f} | best line L{idx} (confidence {conf:.2f}): {PAGE[idx]}\n")

how do I stop getting charged every month?
   answerable 0.98 | best line L1 (confidence 1.00): You can terminate your subscription at any time from Settings > Billing.

is there a student discount?
   answerable 0.02 | best line L7 (confidence 0.55): Need a hand? Reach support in-app or at help@nimbus.example.



**What to notice**
- *"stop getting charged every month"* finds *"terminate your subscription"*, with no shared keywords, and the gate is
  near **1**.
- For the student discount, the `Choice` **still picks a line** (it has to), but the `answerable` gate is near **0** and
  the `Choice` confidence is clearly lower. Without the gate, we'd hand that irrelevant line to the LLM.
- Both answers came from **one** request.

## Step 3 · OpenAI answers, grounded in that one line

`find()` is the whole pipeline:
1. ask Jev (`retrieve`),
2. if `answerable < ANSWERABLE_AT`, **return early**: no LLM call, no cost, no made-up answer,
3. otherwise send OpenAI **only the winning line** with the instruction *"Using ONLY this text, answer in one short
   sentence"*.

Keeping the LLM's input to one line is the grounding: it has nothing else to draw on.
This cell only defines the function.

In [5]:
def find(query, client, oai):
    idx, answerable, conf = retrieve(query, client)
    if answerable < ANSWERABLE_AT:
        return None, answerable, conf, None      # not on this page: skip the LLM entirely

    chat = oai.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content":
                   f"Using ONLY this text, answer in one short sentence.\nText: {PAGE[idx]}\nQuestion: {query}"}],
        max_completion_tokens=2000,
    ).choices[0].message.content.strip()
    return idx, answerable, conf, chat

## Step 4 · Run it

Four questions, none phrased the way the page phrases it. For each, we show the line Jev picked and the reply.

In [6]:
for q in ["how do I stop getting charged every month?", "can I take my drawings out of here?",
          "does this run on my phone?", "is there a student discount?"]:
    idx, answerable, conf, chat = find(q, jev, oai)
    print(f"🔎 {q}")
    if idx is None:
        print(f"   not found on this page (answerable {answerable:.2f}), no LLM call made\n")
    else:
        print(f"   line L{idx}: {PAGE[idx]}")
        print(f"   💬 {chat}\n")

🔎 how do I stop getting charged every month?
   line L1: You can terminate your subscription at any time from Settings > Billing.
   💬 Terminate your subscription at any time from Settings > Billing.



🔎 can I take my drawings out of here?
   line L3: Export any board to PDF or PNG from the share menu in the top right.
   💬 Yes, you can export any board to PDF or PNG from the share menu in the top right.



🔎 does this run on my phone?
   line L5: Nimbus works in any modern browser; there is no separate app to install.
   💬 Yes—Nimbus works in any modern browser, so it should run on your phone if your phone has a modern browser.

🔎 is there a student discount?
   not found on this page (answerable 0.02), no LLM call made



**What to notice**
- *"take my drawings out"* lands on the **export** line and *"run on my phone"* on the **any modern browser** line:
  matching by meaning, not by words.
- Each reply only restates its line. For the phone question the page never says "phone", so a good reply stays
  careful (it works in a browser) rather than promising a mobile app.
- The student-discount question stops at the gate: **no OpenAI call** is made, so it costs only the Jev call and can't
  hallucinate a discount.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a text box wired to `find()` with `ipywidgets`,
and an HTML view that highlights the winning line on the page and shows the reply as a chat bubble. The retrieval
logic above is unchanged.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

# small style kit
INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:820px;">{inner}</div>'


def text_app(placeholder, render, button="Run", value=""):
    """A text box + button. On submit, call render(text) and show the HTML it returns."""
    inp = widgets.Text(value=value, placeholder=placeholder, continuous_update=False,
                       layout=widgets.Layout(width="68%"))
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            if inp.value.strip():
                display(HTML(box("<i>thinking...</i>")))
                html = render(inp.value.strip())
                out.clear_output(wait=True)
                display(HTML(html))

    btn.on_click(go)
    inp.observe(go, names="value")   # fires when you press Enter
    display(widgets.HBox([inp, btn]), out)

`find_html()` renders the whole page with Jev's chosen line highlighted, like a real Ctrl-F, plus either the chat reply
or a *"not found"* note.

In [8]:
def find_html(query):
    idx, answerable, conf, chat = find(query, jev, oai)
    lines = "".join(
        f'<div style="padding:4px 6px;background:#FCEFB4;border-radius:4px;font-weight:600;">{line}</div>' if i == idx
        else f'<div style="padding:4px 6px;color:{SLATE};">{line}</div>'
        for i, line in enumerate(PAGE))
    if idx is None:
        answer = f'<div style="color:{CORAL};margin-top:10px;">not found on this page (answerable {answerable:.2f})</div>'
    else:
        answer = (f'<div style="margin-top:12px;padding:10px 14px;background:{CORAL};color:white;border-radius:14px;'
                  f'display:inline-block;">&#128172; {chat}</div>')
    return box(f'<div style="{CARD}"><div style="font-size:13px;color:{SLATE};">&#128269; {query}</div>'
               f'<div style="margin-top:8px;">{lines}</div>{answer}</div>')


display(HTML(find_html("how do I stop getting charged every month?")))

## Step 6 · Try it yourself

Ask the Nimbus help page anything. Press **Enter** or click **Find** (needs a live kernel). Good ones to try:
- `how many boards can I make for free?` → highlights the **Free plan** line.
- `will you share my info with advertisers?` → highlights the **we never sell your data** line.
- `is there an iPad app?` → highlights *"no separate app to install"*: the answer is *no*, and it is on the page.
- `can I use this offline?` → probably **not found**: the page says it runs in a browser but never mentions offline use.

In [9]:
text_app("ask the Nimbus help page anything, e.g. 'can I use this offline?'", find_html, "Find")

Output()

## Recap and exercises

**Key ideas**
1. A `Choice` can range over **your data**: build `criteria` from the lines of a page at run time.
2. A `Choice` always picks something. Pair it with a `Noul` gate so the system can say *"not here"*.
3. **Jev decides, the LLM phrases.** Retrieval and gating are cheap typed judgments; the LLM only sees the one line it needs.
4. Returning early when the gate says no saves money **and** prevents hallucinated answers.

**Exercises**
- Add a line to `PAGE` (e.g. *"Students get 50% off with a .edu email"*) and re-ask the student-discount question.
- Return the **top two** lines (from `best.probabilities`) to the LLM when the `Choice` confidence is low.
- Raise `ANSWERABLE_AT` to 0.8. Which of the try-it questions now come back "not found"?
- Replace `PAGE` with the paragraphs of a real help page you use, and try five questions that Ctrl-F would miss.